In [1]:
import pandas as pd
import numpy as np

raw_path = "NOCRCWQ.csv"

df = pd.read_csv(
    raw_path,
    skiprows=2,
    low_memory=False
)

df["DateTimeStamp"] = pd.to_datetime(df["DateTimeStamp"])

def parse_flag(s):
    if not isinstance(s, str):
        return np.nan
    if "<" in s and ">" in s:
        start = s.find("<") + 1
        end = s.find(">", start)
        try:
            return int(s[start:end])
        except ValueError:
            return np.nan
    return np.nan

vars_to_clean = ["Temp", "SpCond", "Sal", "DO_pct",
                 "DO_mgl", "Depth", "cDepth", "pH",
                 "Turb", "ChlFluor"]

for v in vars_to_clean:
    f_col = f"F_{v}"
    code_col = f"{f_col}_code"
    if f_col in df.columns:
        df[code_col] = df[f_col].apply(parse_flag)
        df.loc[df[code_col] != 0, v] = np.nan

drop_cols = [c for c in df.columns
             if c.startswith("F_") and not c.endswith("_code")]
if "Unnamed: 26" in df.columns:
    drop_cols.append("Unnamed: 26")

df_clean_15 = df.drop(columns=drop_cols)
df_clean_15 = (
    df_clean_15
    .sort_values("DateTimeStamp")
    .drop_duplicates(subset=["DateTimeStamp"])
)

df_daily_wq = (
    df_clean_15
    .set_index("DateTimeStamp")
    .resample("D")
    .mean(numeric_only=True)
    .reset_index()
)

df_daily_wq["date"] = df_daily_wq["DateTimeStamp"].dt.date
cols_keep = ["date"] + [v for v in vars_to_clean if v in df_daily_wq.columns]
df_daily_wq = df_daily_wq[cols_keep]

out_path = "NOCRCWQ_daily_WQ_clean.csv"
df_daily_wq.to_csv(out_path, index=False)

print("15min 清洗后形状:", df_clean_15.shape)
print("日均水质形状:", df_daily_wq.shape)
print("已保存为:", out_path)
print(df_daily_wq.head())


15min 清洗后形状: (382848, 25)
日均水质形状: (3988, 11)
已保存为: NOCRCWQ_daily_WQ_clean.csv
         date       Temp     SpCond        Sal     DO_pct    DO_mgl     Depth  \
0  2014-01-01  12.388542  50.851042  33.298958  92.768750  8.051042  1.425625   
1  2014-01-02  12.536458  50.814479  33.276042  90.650000  7.836458  1.325937   
2  2014-01-03  11.251042  50.358646  32.887500  94.522917  8.434375  1.263437   
3  2014-01-04   9.614583  52.135208  34.056250  97.419792  8.933333  1.607708   
4  2014-01-05  11.014583  52.043958  34.091667  97.355208  8.653125  1.603021   

   cDepth        pH      Turb  ChlFluor  
0     NaN  7.969792  5.729167       NaN  
1     NaN  7.971875  5.178947       NaN  
2     NaN  7.986458  5.572917       NaN  
3     NaN  8.064583  3.625000       NaN  
4     NaN  8.007292  2.031250       NaN  


In [1]:
import pandas as pd

wq = pd.read_csv("NOCRCWQ_daily_WQ_clean.csv")
precip = pd.read_csv("NC_precip_daily_2014_2024.csv")
chl = pd.read_csv("NC_chlorophyll_daily_2014_2024.csv")

print("WQ columns:", wq.columns)
print("Precip columns:", precip.columns)
print("Chl columns:", chl.columns)
wq.head(), precip.head(), chl.head()


WQ columns: Index(['date', 'Temp', 'SpCond', 'Sal', 'DO_pct', 'DO_mgl', 'Depth', 'cDepth',
       'pH', 'Turb', 'ChlFluor'],
      dtype='object')
Precip columns: Index(['time', 'precip'], dtype='object')
Chl columns: Index(['date', 'chlorophyll'], dtype='object')


(         date       Temp     SpCond        Sal     DO_pct    DO_mgl     Depth  \
 0  2014-01-01  12.388542  50.851042  33.298958  92.768750  8.051042  1.425625   
 1  2014-01-02  12.536458  50.814479  33.276042  90.650000  7.836458  1.325937   
 2  2014-01-03  11.251042  50.358646  32.887500  94.522917  8.434375  1.263437   
 3  2014-01-04   9.614583  52.135208  34.056250  97.419792  8.933333  1.607708   
 4  2014-01-05  11.014583  52.043958  34.091667  97.355208  8.653125  1.603021   
 
    cDepth        pH      Turb  ChlFluor  
 0     NaN  7.969792  5.729167       NaN  
 1     NaN  7.971875  5.178947       NaN  
 2     NaN  7.986458  5.572917       NaN  
 3     NaN  8.064583  3.625000       NaN  
 4     NaN  8.007292  2.031250       NaN  ,
          time     precip
 0  2014-01-01   1.740819
 1  2014-01-02  22.874502
 2  2014-01-03   0.806732
 3  2014-01-04   0.000000
 4  2014-01-05  11.715663,
          date  chlorophyll
 0  2014-01-03     3.115033
 1  2014-01-04     2.397363
 2  20

In [2]:
wq["date"] = pd.to_datetime(wq["date"])
precip["date"] = pd.to_datetime(precip["time"])
chl["date"] = pd.to_datetime(chl["date"])

wq = wq.sort_values("date").reset_index(drop=True)
precip = precip.sort_values("date").reset_index(drop=True)
chl = chl.sort_values("date").reset_index(drop=True)

wq["date"].min(), wq["date"].max(), precip["date"].min(), precip["date"].max(), chl["date"].min(), chl["date"].max()


(Timestamp('2014-01-01 00:00:00'),
 Timestamp('2024-12-01 00:00:00'),
 Timestamp('2014-01-01 00:00:00'),
 Timestamp('2024-12-01 00:00:00'),
 Timestamp('2014-01-03 00:00:00'),
 Timestamp('2024-12-01 00:00:00'))

In [3]:
precip = precip.rename(columns={"precip": "P"})

precip["P"] = precip["P"].astype(float)

precip = precip.sort_values("date").reset_index(drop=True)
precip["P_lag1"] = precip["P"].shift(1)
precip["P_lag2"] = precip["P"].shift(2)
precip["P_3d"] = precip["P"].rolling(window=3, min_periods=1).sum()
precip["P_7d"] = precip["P"].rolling(window=7, min_periods=1).sum()

p95 = precip["P"].quantile(0.95)
precip["P_extreme"] = (precip["P"] > p95).astype(int)

precip_features_path = "NC_precip_features_daily_2014_2024.csv"
precip.to_csv(precip_features_path, index=False)

precip.head(), precip.describe()[["P", "P_3d", "P_7d"]]


(         time          P       date     P_lag1     P_lag2       P_3d  \
 0  2014-01-01   1.740819 2014-01-01        NaN        NaN   1.740819   
 1  2014-01-02  22.874502 2014-01-02   1.740819        NaN  24.615321   
 2  2014-01-03   0.806732 2014-01-03  22.874502   1.740819  25.422053   
 3  2014-01-04   0.000000 2014-01-04   0.806732  22.874502  23.681234   
 4  2014-01-05  11.715663 2014-01-05   0.000000   0.806732  12.522395   
 
         P_7d  P_extreme  
 0   1.740819          0  
 1  24.615321          1  
 2  25.422053          0  
 3  25.422053          0  
 4  37.137716          0  ,
                  P         P_3d         P_7d
 count  3988.000000  3988.000000  3988.000000
 mean      4.110991    12.332257    28.761048
 min       0.000000     0.000000     0.000000
 25%       0.032190     1.804641    12.372261
 50%       1.105917     7.772991    24.846930
 75%       5.471994    18.039911    40.108890
 max      61.026897   111.630999   153.385887
 std       6.764569    13.903

In [4]:
wq["date"] = pd.to_datetime(wq["date"])
precip["date"] = pd.to_datetime(precip["date"])

data = pd.merge(
    wq,
    precip[["date", "P", "P_lag1", "P_lag2", "P_3d", "P_7d", "P_extreme"]],
    on="date",
    how="inner"
)

data = data.sort_values("date").reset_index(drop=True)

main_data_path = "WQ_Precip_main_daily.csv"
data.to_csv(main_data_path, index=False)

data.shape, data.head()


((3988, 17),
         date       Temp     SpCond        Sal     DO_pct    DO_mgl     Depth  \
 0 2014-01-01  12.388542  50.851042  33.298958  92.768750  8.051042  1.425625   
 1 2014-01-02  12.536458  50.814479  33.276042  90.650000  7.836458  1.325937   
 2 2014-01-03  11.251042  50.358646  32.887500  94.522917  8.434375  1.263437   
 3 2014-01-04   9.614583  52.135208  34.056250  97.419792  8.933333  1.607708   
 4 2014-01-05  11.014583  52.043958  34.091667  97.355208  8.653125  1.603021   
 
    cDepth        pH      Turb  ChlFluor          P     P_lag1     P_lag2  \
 0     NaN  7.969792  5.729167       NaN   1.740819        NaN        NaN   
 1     NaN  7.971875  5.178947       NaN  22.874502   1.740819        NaN   
 2     NaN  7.986458  5.572917       NaN   0.806732  22.874502   1.740819   
 3     NaN  8.064583  3.625000       NaN   0.000000   0.806732  22.874502   
 4     NaN  8.007292  2.031250       NaN  11.715663   0.000000   0.806732   
 
         P_3d       P_7d  P_extreme

In [5]:
chl = chl.rename(columns={"chlorophyll": "chl_sat"})
chl["date"] = pd.to_datetime(chl["date"])
chl = chl.sort_values("date").reset_index(drop=True)

data_full = pd.merge(
    data,
    chl[["date", "chl_sat"]],
    on="date",
    how="left"
)

full_path = "WQ_Precip_Chl_full_daily.csv"
data_full.to_csv(full_path, index=False)

data_full.shape, data_full.head()


((3988, 18),
         date       Temp     SpCond        Sal     DO_pct    DO_mgl     Depth  \
 0 2014-01-01  12.388542  50.851042  33.298958  92.768750  8.051042  1.425625   
 1 2014-01-02  12.536458  50.814479  33.276042  90.650000  7.836458  1.325937   
 2 2014-01-03  11.251042  50.358646  32.887500  94.522917  8.434375  1.263437   
 3 2014-01-04   9.614583  52.135208  34.056250  97.419792  8.933333  1.607708   
 4 2014-01-05  11.014583  52.043958  34.091667  97.355208  8.653125  1.603021   
 
    cDepth        pH      Turb  ChlFluor          P     P_lag1     P_lag2  \
 0     NaN  7.969792  5.729167       NaN   1.740819        NaN        NaN   
 1     NaN  7.971875  5.178947       NaN  22.874502   1.740819        NaN   
 2     NaN  7.986458  5.572917       NaN   0.806732  22.874502   1.740819   
 3     NaN  8.064583  3.625000       NaN   0.000000   0.806732  22.874502   
 4     NaN  8.007292  2.031250       NaN  11.715663   0.000000   0.806732   
 
         P_3d       P_7d  P_extreme